###Section F

In [0]:
dbutils.widgets.text("catalog", "vp_ms20010501")
catalog = dbutils.widgets.get("catalog")

from pyspark.sql import functions as F, Window
from pyspark.sql.types import StringType, DecimalType, IntegerType, DateType, TimestampType
from pyspark.sql.functions import udf

def id_validations(id):
    if id is None:
        return None
    value = str(id).strip().upper()
    return None if value == "" else value

validate_id = udf(id_validations, StringType())

###F1

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.ops.time_travel_check AS
SELECT 
    COALESCE(curr.charger_id, prev.charger_id) AS charger_id,
    prev.max_kw AS max_kw_before_batch2,
    curr.max_kw AS max_kw_current,
    prev.is_active AS is_active_before_batch2,
    curr.is_active AS is_active_current
FROM (SELECT charger_id, max_kw, is_active FROM {catalog}.silver.chargers VERSION AS OF 1 WHERE charger_id IN ('CH004', 'CH011', 'CH027')) prev
FULL OUTER JOIN (SELECT charger_id, max_kw, is_active FROM {catalog}.silver.chargers WHERE charger_id IN ('CH004', 'CH011', 'CH027')) curr
ON prev.charger_id = curr.charger_id;
""")

###F2

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.ops.charger_changes_batch2 AS
SELECT 
    charger_id,
    _change_type AS change_type,
    max_kw,
    is_active,
    _commit_version AS commit_version
FROM table_changes('{catalog}.silver.chargers', 2, 2);
""")

###F3

In [0]:
spark.sql(f"""
ALTER TABLE {catalog}.gold.fact_sessions CLUSTER BY (date_key, station_id);
""")

spark.sql(f"""
OPTIMIZE {catalog}.gold.fact_sessions;
""")

###F4

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.ops.sessions_drill DEEP CLONE {catalog}.silver.sessions;
""")

spark.sql(f"""
DELETE FROM {catalog}.ops.sessions_drill WHERE charger_id = 'CH001';
""")

restore_version = spark.sql(f"DESCRIBE HISTORY {catalog}.ops.sessions_drill").agg(F.max("version")).collect()[0][0] - 1

spark.sql(f"""
RESTORE TABLE {catalog}.ops.sessions_drill TO VERSION AS OF {restore_version};
""")